# Point cloud classification

Set input and output paths and GEMINI_API_KEY.

In [ ]:
# This code is the classifier with the geometrie feautures, direkt from an input folder filled with .ply point cloud data

In [ ]:
import os
import time
import base64
import numpy as np
import pandas as pd
import open3d as o3d
from PIL import Image
import google.generativeai as genai

#Gemini API
genai.configure(api_key=os.getenv("GEMINI_API_KEY", ""))

# Input/Output Folders
input_dir = r"XXXXX"
output_dir = r"XXXXX"
os.makedirs(output_dir, exist_ok=True)

image_size = (512, 512)


view_angles = {
    "iso_front_right_top": [45, 35.264],
    "iso_back_left_bottom": [225, -35.264],
    "iso_front_left_top": [135, 35.264],
    "iso_back_right_bottom": [315, -35.264]
}


def extract_geometry_features(pcd):
    bbox = pcd.get_axis_aligned_bounding_box()
    extent = bbox.get_extent()
    return {
        "height": extent[2],
        "width": extent[0],
        "depth": extent[1]
    }


def create_isometric_views(pcd_path, output_img_dir):
    os.makedirs(output_img_dir, exist_ok=True)
    pcd = o3d.io.read_point_cloud(pcd_path)

    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False, width=image_size[0], height=image_size[1])
    vis.add_geometry(pcd)
    view_control = vis.get_view_control()

    bbox = pcd.get_axis_aligned_bounding_box()
    center = bbox.get_center()

    image_paths = []
    for name, (azim, elev) in view_angles.items():
        front = [
            np.cos(np.radians(azim)) * np.cos(np.radians(elev)),
            np.sin(np.radians(azim)) * np.cos(np.radians(elev)),
            np.sin(np.radians(elev))
        ]
        view_control.set_zoom(0.7)
        view_control.set_front(front)
        view_control.set_lookat(center)
        view_control.set_up([0, 0, 1])
        vis.poll_events()
        vis.update_renderer()

        img_path = os.path.join(output_img_dir, f"{name}.png")
        vis.capture_screen_image(img_path)
        image_paths.append(img_path)

    vis.destroy_window()
    return image_paths, pcd

# Fixed class list for classification
class_list = [
    "3D-printer", "beamer", "cardboardbox", "chair", "desktop", "heater",
    "lamp", "robotarm", "robotdog", "shelf", "table", "window"
]

# Prompt 
def gemini_classify(images_paths, features, object_name):
    prompt = (
        "You are shown four isometric views of a 3D object. "
        "The object has the following geometric properties (note: the height, width, and depth are derived from the object's axis-aligned bounding box, "
        "and may not represent precise physical dimensions but give a general sense of size and proportions):\n"
        f"- Height: {features['height']:.2f} m\n"
        f"- Width: {features['width']:.2f} m\n"
        f"- Depth: {features['depth']:.2f} m\n\n"
        "Based on the isometric views and the geometric dimensions, choose exactly one word from the list below that best describes the object. "
        "Respond with only one word from this list:\n\n"
        f"{', '.join(sorted(class_list))}"
    )

    try:
        model = genai.GenerativeModel("gemini-1.5-pro")
        images = [Image.open(p) for p in images_paths]

        response = model.generate_content(
            [prompt] + images,
            generation_config={"max_output_tokens": 10}
        )
        return response.text.strip()
    except Exception as e:
        return f"Error: {str(e)}"

# Run classification on all .ply files in the input directory
results = []

for file in os.listdir(input_dir):
    if not file.lower().endswith(".ply"):
        continue

    object_name = os.path.splitext(file)[0]
    print(f"Processing: {file}")

    ply_path = os.path.join(input_dir, file)
    image_output_dir = os.path.join(output_dir, object_name)

    images, pcd = create_isometric_views(ply_path, image_output_dir)
    print("Isometric views created.")

    features = extract_geometry_features(pcd)
    print(f"Geometry → Height: {features['height']:.2f} m, Width: {features['width']:.2f} m, Depth: {features['depth']:.2f} m")

    classification = gemini_classify(images, features, object_name)
    print(f"Classification: {classification}")

    results.append({
        "object": object_name,
        "class": classification
    })

# Export results to CSV
df = pd.DataFrame(results)
csv_output_path = os.path.join(output_dir, "classification_with_geometry.csv")
df.to_csv(csv_output_path, index=False, encoding="utf-8")
print(f"Classification completed. Results saved to: {csv_output_path}")
